<a href="https://colab.research.google.com/github/TordH95/Cylic_Task_Hello_World/blob/main/INFO284_Del_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
import random
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from torchvision import models
from PIL import Image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Kobler notebook til google drive. Bildene blir importert gjennom google drive fordi det ble vurdert som den raskeste og enkleste måten.

In [ ]:
print(os.listdir("/content/drive/MyDrive/INFO284/Data"))

['Art_shuffled', 'Split_data', 'Egne_bilder']


In [ ]:
# TRENGER IKKE Å VÆRE MED. Bare for å sjekke at riktig filepath er valgt.

In [ ]:
source_dir = "/content/drive/MyDrive/INFO284/Data/Art_shuffled"
target_dir = "/content/drive/MyDrive/INFO284/Data/Split_data"

In [ ]:
# Filepath til datasettet og mappen der de splittede dataene skal lagres.

In [ ]:
print(os.listdir(source_dir))

['AiArtData', 'RealArt']


In [ ]:
# TRENGER IKKE Å VÆRE MED. Bare for å se at mappen er delt inn riktig.

In [ ]:
random.seed(42)

if os.path.exists(os.path.join(target_dir, "train")):
    print("Data er allerede splittet, går videre")
else:
    print("Splitter data")

    splits = ["train", "val", "test"]
    ratios = [0.7, 0.15, 0.15]

    classes = os.listdir(source_dir)

    for cls in classes:
        cls_path = os.path.join(source_dir, cls)

        images = [
            img for img in os.listdir(cls_path)
            if img.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]

        random.shuffle(images)

        n_total = len(images)
        n_train = int(ratios[0] * n_total)
        n_val = int(ratios[1] * n_total)

        split_data = {
            "train": images[:n_train],
            "val": images[n_train:n_train+n_val],
            "test": images[n_train+n_val:]
        }

        for split in splits:
            split_dir = os.path.join(target_dir, split, cls)
            os.makedirs(split_dir, exist_ok=True)

            for img in split_data[split]:
                shutil.copy(
                    os.path.join(cls_path, img),
                    os.path.join(split_dir, img)
                )

    print("Ferdig")

Data er allerede splittet, går videre


In [ ]:
# Splitter data i train, val og test med forhold 70/15/15.
# Treningsdata brukes til å lære modellen, valideringsdata brukes til å evaluere modellen under trening,
# og testdata brukes til å måle hvor godt modellen generaliserer til nye data.
# Dataene legges i separate mapper fordi PyTorch (ImageFolder) forventer denne strukturen,
# og det gjør det enklere å håndtere og organisere datasettene.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder(os.path.join(target_dir, "train"), transform=train_transform)
val_ds   = datasets.ImageFolder(os.path.join(target_dir, "val"), transform=val_transform)
test_ds  = datasets.ImageFolder(os.path.join(target_dir, "test"), transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print(train_ds.classes)
print(train_ds.class_to_idx)
print(len(train_ds), len(val_ds), len(test_ds))

['AiArtData', 'RealArt']
{'AiArtData': 0, 'RealArt': 1}
525 112 115


In [ ]:
# Definerer transforms for å standardisere bildene før trening.
# Treningsdata får data augmentation (random flip) for å gjøre modellen mer robust.
# ImageFolder brukes til å lese bilder fra mapper og automatisk tildele labels basert på mappenavn.
# DataLoader brukes til å dele data i batches og gjøre treningen mer effektiv.
# Shuffle brukes kun på treningsdata for å unngå at modellen lærer rekkefølgen på dataene.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device) # Printer om det kjøres på cpu eller gpu

model = models.efficientnet_b0(pretrained=True)

# Bytter til 2 klasser
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)

model = model.to(device)

Device: cuda


In [ ]:
# Vi bruker EfficientNet-B0, som er en pretrained convolutional neural network (CNN).
# Modellen er trent på ImageNet og har allerede lært å gjenkjenne generelle visuelle mønstre
# som kanter, teksturer og former.

# Ved å bruke en pretrained modell kan vi utnytte denne kunnskapen (transfer learning)
# i stedet for å trene en modell fra bunnen av, noe som er spesielt nyttig når datasettet er relativt lite.

# Det siste laget i modellen byttes ut slik at modellen passer til vår oppgave,
# som er en binær klassifikasjon (AI vs human), i stedet for de 1000 klassene i ImageNet.

# Modellen bruker GPU hvis tilgjengelig ettersom at cpu tar langt tid.

# EfficientNet-B0 ble valgt fordi det er en pretrained cnn som er trent på ImageNet. B0 versjonen ble valgt etter testing av
# større modeller uten bedre resultater. Større modeller er også mer ressurskrevende.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# CrossEntropyLoss brukes som tapsfunksjon fordi den er egnet for klassifikasjonsoppgaver.
# Den måler forskjellen mellom modellens prediksjoner og de riktige labelene.

# Adam optimizer brukes til å oppdatere modellens vekter under trening.
# Den er valgt fordi den gir stabil og effektiv læring.

# Learning rate bestemmer hvor store steg modellen tar under oppdatering av vektene.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# train_epoch trener modellen én epoch ved å iterere gjennom treningsdata.
# For hver batch utføres en forward pass (prediksjon), loss beregnes,
# og modellen oppdateres ved hjelp av backpropagation (loss.backward og optimizer.step).

# Accuracy beregnes ved å sammenligne prediksjoner med riktige labels.

# evaluate brukes til å evaluere modellen på validerings- eller testdata.
# Her brukes model.eval() og torch.no_grad() for å deaktivere læring,
# slik at vi kun måler modellens ytelse uten å oppdatere vektene.

In [ ]:
num_epochs = 20
early_stop = 3

best_val_loss = float("inf")
epochs_without_improvement = 0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")
    print(f"Val loss:   {val_loss:.4f}, Val acc:   {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("Val loss improved")
    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement} iterations")

    if epochs_without_improvement >= early_stop:
        print("Stopping")
        break

    print()

Epoch 1/20
Train loss: 0.6693, Train acc: 0.5695
Val loss:   0.6062, Val acc:   0.7411
Val loss improved

Epoch 2/20
Train loss: 0.5243, Train acc: 0.8190
Val loss:   0.5383, Val acc:   0.7679
Val loss improved

Epoch 3/20
Train loss: 0.3916, Train acc: 0.9010
Val loss:   0.4730, Val acc:   0.7946
Val loss improved

Epoch 4/20
Train loss: 0.2979, Train acc: 0.9390
Val loss:   0.4293, Val acc:   0.8036
Val loss improved

Epoch 5/20
Train loss: 0.1985, Train acc: 0.9619
Val loss:   0.4062, Val acc:   0.8393
Val loss improved



In [ ]:
# Trener modellen over flere epochs (her 5 ganger gjennom hele datasettet).
# For hver epoch kjøres både trening (train_epoch) og evaluering (evaluate).

# Train loss og accuracy viser hvordan modellen lærer på treningsdata,
# mens validation loss og accuracy viser hvordan modellen presterer på data den ikke har sett.

# Resultatene lagres i lister for å kunne analyseres senere.
# Verdiene printes også for å følge utviklingen underveis og oppdage eventuell overfitting.

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# Evaluerer modellen på test-settet, som består av data modellen ikke har sett tidligere.
# Dette gir en objektiv måling av modellens evne til å generalisere til nye bilder.
# Test accuracy brukes som endelig resultat for modellen.

In [ ]:
if os.path.exists("new_images"):
  shutil.rmtree("new_images")

os.makedirs("new_images", exist_ok=True)

In [ ]:
# Her begynner del b av oppgaven.
# Det opprettes en mappe i colab for de nye bildene.
# Bildene lagres i mappen, så bildene slettes hver gang denne koden kjøres dersom man
# vil prøve med forskjellige bilder uten å ha mer enn 5 bilder om gangen.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Brukes til å importere bildene lokalt fra pc.

In [ ]:
for file in os.listdir("/content"):
    if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        shutil.move(file, os.path.join("new_images", file))

In [ ]:
# Går gjennom alle filer i Colab-mappen (/content) og flytter alle bildefiler
# til mappen "new_images". Dette gjøres for å samle de nye testbildene på ett sted,
# slik at kun disse brukes i klassifiseringen.

In [ ]:
print(os.listdir("new_images"))

In [ ]:
model.eval()

predict_transform = val_transform
class_names = train_ds.classes

image_folder = "new_images"

results = []

for img_name in os.listdir(image_folder):

    if "(1)" in img_name:   # Duplikater tas ikke med dersom de finnes
        continue

    if img_name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        img_path = os.path.join(image_folder, img_name)

        image = Image.open(img_path).convert("RGB")
        image = predict_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(image)
            probs = torch.softmax(output, dim=1)
            pred_idx = torch.argmax(probs, dim=1).item()

            label_map = {
            "AiArtData": "Ai_generated",
            "RealArt": "Human_art"
        }

        predicted_class = class_names[pred_idx]
        new_names = label_map[predicted_class]
        confidence = probs[0][pred_idx].item()

        print(f"{img_name}: {new_names} ({confidence:.2f})")

In [ ]:
# Bruker den ferdigtrente modellen til å klassifisere nye bilder.
# Modellen lærer ikke, bare evaluerer.
# Bildene lastes inn fra en egen mappe og bearbeides på samme måte som tidligere.

# For hvert bilde gjør modellen en prediksjon, og resultatet konverteres til sannsynligheter.
# Klassen med høyest sannsynlighet velges som modellens prediksjon.

# Noen ganger ble dupliketer av bildene tatt med. Disse ignoreres
# slik at bare de 5 originale bildene blir evaluert.

# Modellen gjør det generelt dårlig på disse bildene. Det skyldes trolig at EfficientNet-B0 er dårlig på å klassifisere
# realistiske AI bilder.